# Make anndata objects from mask filt means 

Run using `aging_thymus` conda environment

In [1]:
import anndata as ad 
from anndata import AnnData
import os
import re
import pandas as pd 
import scanpy as sc
import numpy as np 
from typing import Union, List
from pathlib import Path

# Function

In [8]:
def make_adata_from_mask_filt_means(
    img_dir: Union[Path, str], 
    file_name: str, 
    marker_cols: List[str] = [
        'b-Catenin1', 'Pan-Cytokeratin', 'K5', 'K8', 'K10', 'K14', 'TP63', 
        'CD68', 'CD44', 'CD31', 'Collagen IV', 'Vimentin',
        'HLA-A', 'HLA-DR',
        'Bcl-2', 'CD3e', 'CD4', 'CD8','Ki67', 'FOXP3',  
        'CD20', 'CD11c', 'CD141', 'CD34', 'CD40'],
    do_leiden: bool = False
    ) -> AnnData:
    mean_mask_quant_df = pd.read_csv(os.path.join(img_dir, file_name))
    mean_mask_quant_df.columns = mean_mask_quant_df.columns.str.replace("_mask", "")
    mean_ad = ad.AnnData(X= mean_mask_quant_df.loc[:, marker_cols].values, obs= mean_mask_quant_df.loc[:, ["CellID", "Area", "X_centroid", "Y_centroid"]])
    mean_ad.var_names = marker_cols
    mean_ad = mean_ad[mean_ad.obs["Area"] > 10, :].copy()
    sc.pp.scale(mean_ad)
    mean_ad.X = np.clip(mean_ad.X, -3, 3) # Clip normalized values between -3 and 3                            
    sc.tl.pca(mean_ad)
    sc.pp.neighbors(mean_ad, use_rep= "X_pca")
    if do_leiden:
        for res in [1, 1.5, 1.75]:
            sc.tl.leiden(mean_ad, key_added=f"leiden_{res:4.2f}", resolution=res, flavor="igraph")
            sc.pl.dotplot(mean_ad, groupby=f"leiden_{res:4.2f}", var_names= mean_ad.var_names, title = "mean",  dendrogram = True)
            sc.pl.heatmap(mean_ad, groupby=f"leiden_{res:4.2f}", var_names= mean_ad.var_names,  dendrogram = True)
    mean_ad.write_h5ad(os.path.join(img_dir, re.sub("csv", "h5ad", file_name)))
    return mean_ad

# Making AnnData objects

In [ ]:
mxif_dir="/stor/scratch/Ehrlich/MxIF/aging_thymus/human_images/human_blocks"

# Only keeping tissues with usable stains. Some did not stain well or had no epithelial tissue in focus or no nuclear DAPI
b1_usable_dirs = ["block1_patches/P31_27F", "block1_patches/P32_46F", "block1_patches/P33_10M", "block1_patches/P37_62F", "block1_patches/T2008_7M", "block1_patches/T2033_9F", "block1_patches/T2066_8M"]
b2_usable_dirs = ["block2_patches/P34_29F", "block2_patches/P35_38F", "block2_patches/P36_67F", "block2_patches/T2056-2_0.008M", "block2_patches/T2092-4_0.833M", "block2_patches/T2095-2_22M", "block2_patches/T2101-1_1M"]
b3_usable_dirs = ["block3_patches/T2019-2_0.013F", "block3_patches/T2040-2_8F", "block3_patches/T2044-2_1.166F", "block3_patches/T2087-2_0.003M"]
b4_usable_dirs = ["block4_patches/P41-E1_58F", "block4_patches/P52_21M", "block4_patches/T2018-3_0.3F", "block4_patches/T2036-3_7F", "block4_patches/T2049-1_0.013M", "block4_patches/T2077-1_0.013F", "block4_patches/T2083-1_0.917F"]
b5_usable_dirs = ["block5_patches/T2047-1_0.333F", "block5_patches/T2037-3_1.583M", "block5_patches/A1_20wkM", "block5_patches/T2052-1_0.021M"]
b6_usable_dirs = ["block6_patches/A3_18wkF", "block6_patches/P42_54F", "block6_patches/P44_64F", "block6_patches/P49_2M", "block6_patches/P57_17F", "block6_patches/T2110-1_0.025M"]
usable_dirs = b1_usable_dirs + b2_usable_dirs +  b3_usable_dirs + b4_usable_dirs + b5_usable_dirs + b6_usable_dirs

In [13]:
for usable_dir in usable_dirs:
    usable_dir = os.path.join(mxif_dir, usable_dir)
    quant_name = [file for file in os.listdir(usable_dir) if re.search("thresh_full_auto_thresh_binary_filt_masks_no_touch_nonfat_cell_mask.csv", file)][0]
    img_ad = make_adata_from_mask_filt_means(
        img_dir = os.path.join(mxif_dir, usable_dir),
        file_name = quant_name
    )
    print(f"Done with {usable_dir}")

/stor/work/Ehrlich/Users/John/mamba/envs/aging_thymus/lib/python3.10/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


Done with /stor/scratch/Ehrlich/MxIF/aging_thymus/human_images/human_blocks/block1_patches/P31_27F


/stor/work/Ehrlich/Users/John/mamba/envs/aging_thymus/lib/python3.10/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


Done with /stor/scratch/Ehrlich/MxIF/aging_thymus/human_images/human_blocks/block1_patches/P32_46F


/stor/work/Ehrlich/Users/John/mamba/envs/aging_thymus/lib/python3.10/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


...

Done with /stor/scratch/Ehrlich/MxIF/aging_thymus/human_images/human_blocks/block1_patches/T2008_7M


/stor/work/Ehrlich/Users/John/mamba/envs/aging_thymus/lib/python3.10/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


Done with /stor/scratch/Ehrlich/MxIF/aging_thymus/human_images/human_blocks/block1_patches/T2033_9F


/stor/work/Ehrlich/Users/John/mamba/envs/aging_thymus/lib/python3.10/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


Done with /stor/scratch/Ehrlich/MxIF/aging_thymus/human_images/human_blocks/block1_patches/T2066_8M
